# Cyclistic: how do annual members and casual riders use bikes differently?
Google Data Analytics Capstone, Track A. Phases: **Ask → Prepare → Process → Analyze → Share → Act**.
Heavy lifting runs in **DuckDB (SQL)** directly on the CSV files, so 12 months (~5–6 million trips) fit on an 8 GB laptop; small aggregated results go to **pandas** for charts.

## 1. Ask
- **Business task:** identify how annual members and casual riders use Cyclistic bikes differently, so the marketing team can design a strategy to convert casual riders into annual members.
- **Stakeholders:** Lily Moreno (director of marketing), Cyclistic marketing analytics team, Cyclistic executive team.
- **Metrics:** share of rides, median ride length, weekday vs weekend share, hourly profile, monthly seasonality, bike type mix, top stations, round-trip share.

## 2. Prepare
Data: Divvy trip data 2025-11 → 2026-10 (`python scripts/download_data.py`). License: https://divvybikes.com/data-license-agreement. Details in `docs/data_sources.md`.

In [ ]:
from pathlib import Path
import duckdb, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style="whitegrid")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, CLEAN, FIG = ROOT / "data/raw", ROOT / "data/clean", ROOT / "figures"
CLEAN.mkdir(parents=True, exist_ok=True); FIG.mkdir(exist_ok=True)
con = duckdb.connect(str(CLEAN / "cyclistic.duckdb"))
con.sql("SET memory_limit = '3GB'")
sorted(p.name for p in RAW.glob("*-divvy-tripdata.csv"))

In [ ]:
# Load all 12 monthly files into one table; keep the source file name for checks
con.sql(f'''
CREATE OR REPLACE TABLE trips_raw AS
SELECT * FROM read_csv('{RAW}/*-divvy-tripdata.csv', union_by_name = true, filename = true,
                       types = {{'started_at': 'TIMESTAMP', 'ended_at': 'TIMESTAMP'}})
''')
con.sql("DESCRIBE trips_raw").df()

In [ ]:
# Rows per file and a first look at data quality
con.sql('''
SELECT regexp_extract(filename, '(\\d{6})') AS month_file, count(*) AS rides,
       count(*) FILTER (WHERE start_station_name IS NULL) AS no_start_station,
       count(*) FILTER (WHERE end_lat IS NULL)            AS no_end_coords
FROM trips_raw GROUP BY 1 ORDER BY 1
''').df()

In [ ]:
con.sql("SELECT rideable_type, member_casual, count(*) n FROM trips_raw GROUP BY ALL ORDER BY ALL").df()

## 3. Process
Every step is logged with row counts → copy the table into `docs/changelog.md`.

In [ ]:
steps = []
def log(name, sql_count):
    n = con.sql(sql_count).fetchone()[0]; steps.append((name, n)); print(f"{name:<55} {n:>10,}")

log("raw rows", "SELECT count(*) FROM trips_raw")
log("distinct ride_id", "SELECT count(DISTINCT ride_id) FROM trips_raw")

con.sql('''
CREATE OR REPLACE TABLE trips AS
WITH dedup AS (
  SELECT *, row_number() OVER (PARTITION BY ride_id ORDER BY started_at) AS rn FROM trips_raw
)
SELECT ride_id, rideable_type, member_casual, started_at, ended_at,
       start_station_name, end_station_name, start_lat, start_lng, end_lat, end_lng,
       date_diff('second', started_at, ended_at) / 60.0 AS ride_length_min,
       dayname(started_at)                      AS day_of_week,
       isodow(started_at)                       AS dow,          -- 1 = Monday … 7 = Sunday
       hour(started_at)                         AS hour,
       strftime(started_at, '%Y-%m')            AS month,
       (start_station_name = end_station_name)  AS round_trip
FROM dedup
WHERE rn = 1
  AND member_casual IN ('member', 'casual')
  AND date_diff('second', started_at, ended_at) BETWEEN 60 AND 86400     -- 1 min .. 24 h
  AND coalesce(upper(start_station_name), '') NOT LIKE '%TEST%'
  AND coalesce(upper(end_station_name),   '') NOT LIKE '%TEST%'
  AND coalesce(upper(start_station_name), '') NOT LIKE '%REPAIR%'
''')
log("clean rows (dedup, 1 min–24 h, no test/repair stations)", "SELECT count(*) FROM trips")
con.sql(f"COPY trips TO '{CLEAN}/trips_clean.parquet' (FORMAT parquet)")
pd.DataFrame(steps, columns=["step", "rows"])

## 4. Analyze

In [ ]:
overview = con.sql('''
SELECT member_casual, count(*) AS rides,
       round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct_rides,
       round(median(ride_length_min), 1) AS median_min,
       round(avg(ride_length_min), 1)    AS mean_min,
       round(100.0 * avg(CASE WHEN dow >= 6 THEN 1 ELSE 0 END), 1) AS pct_weekend,
       round(100.0 * avg(CASE WHEN round_trip THEN 1 ELSE 0 END), 1) AS pct_round_trip
FROM trips GROUP BY 1
''').df(); overview

In [ ]:
by_day   = con.sql("SELECT member_casual, dow, day_of_week, count(*) rides, median(ride_length_min) median_min FROM trips GROUP BY ALL ORDER BY dow").df()
by_hour  = con.sql("SELECT member_casual, (dow >= 6) AS weekend, hour, count(*) rides FROM trips GROUP BY ALL ORDER BY hour").df()
by_month = con.sql("SELECT member_casual, month, count(*) rides, median(ride_length_min) median_min FROM trips GROUP BY ALL ORDER BY month").df()
by_type  = con.sql("SELECT member_casual, rideable_type, count(*) rides FROM trips GROUP BY ALL").df()
top_casual = con.sql('''
SELECT start_station_name, count(*) rides, avg(start_lat) lat, avg(start_lng) lng
FROM trips WHERE member_casual = 'casual' AND start_station_name IS NOT NULL
GROUP BY 1 ORDER BY rides DESC LIMIT 15''').df()
top_casual.head(10)

## 5. Share
Each chart title states the finding, not the variable. Save PNGs to `figures/`; export CSVs for Tableau Public.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=by_day, x="day_of_week", y="rides", hue="member_casual", ax=ax)
ax.set(title="<write the finding here>", xlabel="", ylabel="Rides")
fig.tight_layout(); fig.savefig(FIG / "01_rides_by_day.png", dpi=200)

In [ ]:
g = sns.relplot(data=by_hour, x="hour", y="rides", hue="member_casual", col="weekend", kind="line", height=3.5, aspect=1.4)
g.figure.suptitle("<write the finding here>", y=1.05)
g.savefig(FIG / "02_rides_by_hour.png", dpi=200)

In [ ]:
for name, df in {"by_day": by_day, "by_hour": by_hour, "by_month": by_month, "by_type": by_type, "top_casual_stations": top_casual}.items():
    df.to_csv(CLEAN / f"tableau_{name}.csv", index=False)

## 6. Act
**Top three recommendations** (each tied to a finding and a chart above):
1. …
2. …
3. …

**Limitations:** no rider ID (cannot count people or observe conversions), no pricing/revenue data, station names missing on many e-bike trips.